# Notebook 03 — Language Model

**Project:** SciSpell — a domain-aware spelling correction system for scientific text
**Stage:** Turning corpus counts into probabilities

---

## The problem this solves

Edit distance narrows the candidates but cannot pick between them. `acress` is
one edit from *across*, *actress*, *access*, *acres*, and *caress* — all equally
"near". What separates them is **likelihood**: some words are simply more
probable than others, and more probable still in a particular context. In

> "she is an acress and a director"

the word *an* before the typo makes *actress* the obvious reading, while

> "we walked acress the field"

points to *across*. Humans do this instantly; a **language model** is the
machinery that lets the system do it with arithmetic.

## What a language model is

A language model assigns a probability to text. We build two:

- **Unigram model** — P(w): how probable is word *w* on its own?
  Estimated by relative frequency: count(w) / total tokens.
- **Bigram model** — P(w | v): how probable is *w* given that the previous
  word is *v*? Estimated by count(v, w) / count(v).

The bigram model is where context lives — it is what will separate the two
`acress` sentences above, and later (Notebook 04) it is the only tool that can
catch **real-word errors** like *threw* vs *through*, where the typed word is in
the dictionary and only context reveals the mistake.

## The zero problem, and smoothing

Our corpus is 229k tokens; English is unbounded. Most valid word pairs never
occur in any finite corpus — Zipf's long tail guaranteed it. Raw counts would
assign them probability **zero**, and a single zero multiplied into a sentence
score annihilates everything else. **Smoothing** reserves a little probability
mass for unseen events; we implement and compare Laplace (add-one) and add-k,
and justify the choice with perplexity on held-out text.

## What this notebook builds

1. Sentence segmentation with boundary markers `<s>` and `</s>`
2. Unigram and bigram counts and probabilities from the corpus
3. Smoothing, with the vocabulary fixed to the dictionary from Notebook 01
4. A train/held-out split and **perplexity** to evaluate the model honestly
5. Saved model artefacts for the correction engine

In [1]:
# ── Setup — reload artefacts, segment the corpus into sentences ───
import re
import json
import sys
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt

CWD = Path.cwd()
PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD
DATA_DIR = PROJECT_ROOT / "data"
FIG_DIR  = PROJECT_ROOT / "figures"

sys.path.insert(0, str(PROJECT_ROOT / "app"))
from edit_distance import tokenize, TOKEN_RE          # the one true tokenizer

word_freq  = {w: int(c) for w, c in
              json.loads((DATA_DIR / "word_freq.json").read_text(encoding="utf-8")).items()}
DICTIONARY = set((DATA_DIR / "dictionary.txt").read_text(encoding="utf-8").split("\n"))
corpus_text = (DATA_DIR / "science_corpus.txt").read_text(encoding="utf-8")

# ── Sentence segmentation ──
# Split on ., !, ? followed by whitespace; the == banners and blank lines also break sentences.
SENT_SPLIT_RE = re.compile(r"(?<=[.!?])\s+|\n{2,}|={3,}[^\n]*")

BOS, EOS = "<s>", "</s>"     # beginning / end of sentence markers

sentences = []
for chunk in SENT_SPLIT_RE.split(corpus_text):
    toks = tokenize(chunk or "")
    if toks:                                  # skip empty / punctuation-only chunks
        sentences.append([BOS] + toks + [EOS])

sent_lengths = np.array([len(s) - 2 for s in sentences])   # exclude markers

print(f"Sentences          : {len(sentences):,}")
print(f"Tokens (no markers): {int(sent_lengths.sum()):,}   "
      f"(Notebook 01 counted {sum(word_freq.values()):,})")
print(f"Sentence length    : mean {sent_lengths.mean():.1f}, "
      f"median {int(np.median(sent_lengths))}, max {int(sent_lengths.max())}")
print(f"\nExample sentence   : {' '.join(sentences[200])}")

Sentences          : 8,316
Tokens (no markers): 229,325   (Notebook 01 counted 229,325)
Sentence length    : mean 27.6, median 24, max 215

Example sentence   : <s> these facts seemed to me to throw some light on the origin of species that mystery of mysteries as it has been called by one of our greatest philosophers </s>


## 1. Counting: unigrams and bigrams

### Maximum likelihood estimation

The simplest way to turn counts into probabilities is **maximum likelihood
estimation** (MLE) — the estimate that makes the observed corpus as likely as
possible, which turns out to be plain relative frequency:

    P(w)      =  count(w) / N                       ← unigram
    P(w | v)  =  count(v, w) / count(v)             ← bigram

where N is the total number of tokens. The bigram formula reads: *of all the
times word v appeared, what fraction of the time was it followed by w?*

Sentence markers participate in the counts. `<s>` is the "previous word" of every
sentence-initial word, so P(*the* | `<s>`) is a real, useful quantity — the
probability a sentence starts with *the*. Likewise `</s>` lets the model express
how likely a word is to end a sentence.

### The zero problem

MLE has a fatal flaw for our purposes: any word pair that never occurs in the
corpus receives probability exactly **zero**. Not "unlikely" — impossible. Since
sentence probability is a product of bigram probabilities, one zero drives the
entire sentence to zero, and the model can no longer distinguish a slightly
unusual sentence from an absurd one.

This is not a small edge case. With V distinct words there are V² possible
bigrams; our corpus can contain at most N of them. The cell below quantifies how
tiny that fraction is — motivation for the smoothing that follows.

In [2]:
# ── Unigram and bigram counts, MLE, and the zero problem ──────────
unigram_counts = Counter()
bigram_counts  = Counter()

for sent in sentences:
    unigram_counts.update(sent)
    bigram_counts.update(zip(sent[:-1], sent[1:]))

N_tokens = sum(unigram_counts.values())          # includes <s> and </s>
V_corpus = len(unigram_counts)

def p_unigram_mle(w):
    return unigram_counts[w] / N_tokens

def p_bigram_mle(prev, w):
    if unigram_counts[prev] == 0:
        return 0.0
    return bigram_counts[(prev, w)] / unigram_counts[prev]

print(f"Unigram types (incl. markers) : {V_corpus:,}")
print(f"Unigram tokens               : {N_tokens:,}")
print(f"Distinct bigrams observed    : {len(bigram_counts):,}")
print(f"Possible bigrams (V²)        : {V_corpus**2:,}")
print(f"Coverage                     : {len(bigram_counts) / V_corpus**2:.4%}")

print("\nMost frequent bigrams:")
for (v, w), c in bigram_counts.most_common(8):
    print(f"  {v + ' ' + w:<24} {c:>5,}   P({w}|{v}) = {p_bigram_mle(v, w):.4f}")

print("\nSentence-initial preferences — P(w | <s>):")
starters = sorted(((w, c) for (v, w), c in bigram_counts.items() if v == BOS),
                  key=lambda x: -x[1])[:5]
for w, c in starters:
    print(f"  <s> {w:<20} {c:>5,}   {p_bigram_mle(BOS, w):.4f}")

print("\nThe zero problem — plausible English pairs absent from this corpus:")
for v, w in [("scientific", "method"), ("natural", "selection"),
             ("the", "keyboard"), ("quantum", "mechanics"), ("we", "observe")]:
    c = bigram_counts[(v, w)]
    print(f"  P({w:<10}|{v:<10}) = {p_bigram_mle(v, w):.6f}   (count {c})")

Unigram types (incl. markers) : 8,897
Unigram tokens               : 245,957
Distinct bigrams observed    : 83,664
Possible bigrams (V²)        : 79,156,609
Coverage                     : 0.1057%

Most frequent bigrams:
  of the                   3,139   P(the|of) = 0.2735
  in the                   1,583   P(the|in) = 0.2827
  to the                     983   P(the|to) = 0.1813
  the same                   904   P(same|the) = 0.0555
  on the                     837   P(the|on) = 0.4412
  <s> the                    683   P(the|<s>) = 0.0821
  it is                      633   P(is|it) = 0.2674
  that the                   627   P(the|that) = 0.1953

Sentence-initial preferences — P(w | <s>):
  <s> the                    683   0.0821
  <s> i                      468   0.0563
  <s> we                     379   0.0456
  <s> in                     375   0.0451
  <s> but                    345   0.0415

The zero problem — plausible English pairs absent from this corpus:
  P(method    |scient

## 2. Smoothing

**Smoothing** takes a little probability mass away from events we *have* seen and
redistributes it to events we have *not*, so nothing is impossible.

### Laplace (add-one) smoothing

Pretend every possible bigram was seen one extra time:

    P_laplace(w | v)  =  (count(v, w) + 1) / (count(v) + V)

The `+ V` in the denominator keeps the distribution summing to 1, since we added
1 to each of V possible continuations.

### The cost of add-one

Laplace is simple but blunt. With V ≈ 8,900, a context like *of* (3,139
occurrences of *of the* out of 11,476) has its probability mass spread across
8,900 imaginary continuations. The effect is easiest to see through the
**reconstituted count** — what the smoothed probability implies the count
"should have been":

    c*(v, w)  =  (count(v, w) + 1) × count(v) / (count(v) + V)

If a bigram seen 3,139 times is reconstituted as far fewer, Laplace has taken
too much from the evidence we actually have to pay for events we merely imagine.

### Add-k smoothing

The fix is to add a fraction instead of a whole count:

    P_addk(w | v)  =  (count(v, w) + k) / (count(v) + k·V)     with 0 < k < 1

Smaller k respects observed counts more and reserves less for the unseen. But k
is now a **hyperparameter** — it cannot be chosen by taste. Section 3 selects it
by measuring performance on held-out text.

### Vocabulary choice

The language model's vocabulary is the **corpus** vocabulary (plus the two
markers), not the 370k-word dictionary. Spreading smoothing mass over 370k
mostly-irrelevant words would starve the words that actually occur. Dictionary
words absent from the corpus are handled as unknown (`<UNK>`) and receive a
uniform floor probability; in the correction engine it is then the error model
P(x | w) that discriminates among them.

In [3]:
# ── Laplace and add-k smoothing ───────────────────────────────────
V_LM = V_corpus                      # 8,897 — corpus types incl. <s>, </s>

def p_bigram_laplace(prev, w):
    return (bigram_counts[(prev, w)] + 1) / (unigram_counts[prev] + V_LM)

def p_bigram_addk(prev, w, k=0.1):
    return (bigram_counts[(prev, w)] + k) / (unigram_counts[prev] + k * V_LM)

def reconstituted_count(prev, w, k=1.0):
    return (bigram_counts[(prev, w)] + k) * unigram_counts[prev] / (unigram_counts[prev] + k * V_LM)

pairs = [("of", "the"), ("natural", "selection"), ("it", "is"),
         ("the", "same"), ("scientific", "method"), ("the", "keyboard"),
         ("quantum", "mechanics")]

print(f"{'bigram':<24}{'count':>7}{'MLE':>10}{'Laplace':>11}{'add-k .1':>11}{'add-k .01':>11}")
print("─" * 74)
for v, w in pairs:
    print(f"{v + ' ' + w:<24}{bigram_counts[(v, w)]:>7,}"
          f"{p_bigram_mle(v, w):>10.5f}{p_bigram_laplace(v, w):>11.5f}"
          f"{p_bigram_addk(v, w, 0.1):>11.5f}{p_bigram_addk(v, w, 0.01):>11.5f}")

print("\nHow much evidence does add-one destroy?")
print(f"{'bigram':<24}{'observed':>10}{'c* (k=1)':>11}{'c* (k=0.1)':>12}{'kept':>8}")
print("─" * 65)
for v, w in [("of", "the"), ("in", "the"), ("natural", "selection"), ("it", "is")]:
    c   = bigram_counts[(v, w)]
    c1  = reconstituted_count(v, w, 1.0)
    c01 = reconstituted_count(v, w, 0.1)
    print(f"{v + ' ' + w:<24}{c:>10,}{c1:>11.1f}{c01:>12.1f}{c1 / c:>7.0%}")

print("\nUnseen pairs are no longer impossible:")
for v, w in [("scientific", "method"), ("the", "keyboard")]:
    print(f"  P({w}|{v}): MLE {p_bigram_mle(v, w):.6f} → "
          f"add-k(0.1) {p_bigram_addk(v, w, 0.1):.8f}")

bigram                    count       MLE    Laplace   add-k .1  add-k .01
──────────────────────────────────────────────────────────────────────────
of the                    3,139   0.27353    0.15413    0.25386    0.27142
natural selection           287   0.68990    0.03092    0.21988    0.56837
it is                       633   0.26743    0.05629    0.19440    0.25774
the same                    904   0.05546    0.03592    0.05259    0.05516
scientific method             0   0.00000    0.00011    0.00011    0.00011
the keyboard                  0   0.00000    0.00004    0.00001    0.00000
quantum mechanics             0   0.00000    0.00011    0.00011    0.00011

How much evidence does add-one destroy?
bigram                    observed   c* (k=1)  c* (k=0.1)    kept
─────────────────────────────────────────────────────────────────
of the                       3,139     1768.7      2913.2    56%
in the                       1,583      611.9      1366.1    39%
natural selection     

## 3.  Choosing k honestly: held-out data and perplexity


### Why we cannot judge the model on its own training data

Any model looks best on the text it memorised. The honest test is text the model
has **never seen**: we split the sentences 90/10 into a **training set** (counts
come only from here) and a **held-out set** (evaluation happens only here). The
split is shuffled with a fixed random seed so results are reproducible.

### Perplexity

The standard score for a language model is **perplexity** — derived from the
probability the model assigns to the held-out text. For a text of N words:

    PP  =  P(w₁ … w_N) ^ (−1/N)

computed in log-space to avoid numerical underflow:

    PP  =  exp( − (1/N) · Σ log P(wᵢ | wᵢ₋₁) )

Intuition: perplexity is the model's **average branching factor** — "on average,
how many words was the model effectively choosing between at each step?"
A model that always knew the next word exactly would have perplexity 1; a model
guessing uniformly over V words would have perplexity V. **Lower is better.**

### The tournament

We evaluate add-k for k ∈ {1, 0.5, 0.1, 0.05, 0.01, 0.005, 0.001} on the
held-out set and keep the winner. Expect a U-shape: large k over-smooths
(evidence destroyed), tiny k under-smooths (unseen bigrams get punished too
hard, and each one costs a large negative log term). Somewhere between lies the
best trade-off — found by measurement, not preference.

Held-out words absent from the training vocabulary are mapped to `<UNK>` before
scoring, so the model is never asked for the probability of a word it could not
have known exists.

In [4]:
# ── Train/held-out split, perplexity, and the k tournament ────────
rng = np.random.default_rng(42)
order = rng.permutation(len(sentences))
cut = int(0.9 * len(sentences))
train_sents = [sentences[i] for i in order[:cut]]
heldout_sents = [sentences[i] for i in order[cut:]]

# Counts from TRAINING data only
train_uni = Counter()
train_bi  = Counter()
for sent in train_sents:
    train_uni.update(sent)
    train_bi.update(zip(sent[:-1], sent[1:]))

UNK = "<UNK>"
train_vocab = set(train_uni)
V_train = len(train_vocab) + 1                   # +1 for <UNK> itself

def map_unk(sent):
    return [w if w in train_vocab else UNK for w in sent]

def perplexity_addk(sents, k):
    log_sum, n_words = 0.0, 0
    for sent in sents:
        sent = map_unk(sent)
        for prev, w in zip(sent[:-1], sent[1:]):
            p = (train_bi[(prev, w)] + k) / (train_uni[prev] + k * V_train)
            log_sum += np.log(p)
            n_words += 1
    return float(np.exp(-log_sum / n_words))

heldout_tokens = sum(len(s) - 1 for s in heldout_sents)
unk_rate = (sum(1 for s in heldout_sents for w in s if w not in train_vocab)
            / sum(len(s) for s in heldout_sents))

print(f"Training sentences : {len(train_sents):,}   Held-out: {len(heldout_sents):,}")
print(f"Held-out bigram evaluations: {heldout_tokens:,}")
print(f"Held-out <UNK> rate        : {unk_rate:.2%}\n")

ks = [1.0, 0.5, 0.1, 0.05, 0.01, 0.005, 0.001]
results = {k: perplexity_addk(heldout_sents, k) for k in ks}
best_k = min(results, key=results.get)

print(f"{'k':>8}{'held-out perplexity':>22}")
print("─" * 30)
for k, pp in results.items():
    print(f"{k:>8}{pp:>22,.1f}{'   ← best' if k == best_k else ''}")

print(f"\ntrain-set perplexity at k={best_k} : "
      f"{perplexity_addk(train_sents, best_k):,.1f}   (memorisation gap — expected)")

Training sentences : 7,484   Held-out: 832
Held-out bigram evaluations: 24,571
Held-out <UNK> rate        : 1.37%

       k   held-out perplexity
──────────────────────────────
     1.0               1,263.2
     0.5                 918.4
     0.1                 494.0
    0.05                 407.8
    0.01                 318.8
   0.005                 311.8   ← best
   0.001                 348.8

train-set perplexity at k=0.005 : 69.4   (memorisation gap — expected)


## 4. Finalising and exporting the model

Two decisions close the modelling work:

1. **Final counts come from the full corpus.** The 90/10 split existed only to
   choose k honestly. With k = 0.005 now fixed, withholding 10% of the data
   forever would waste evidence — so the shipped model is trained on all
   sentences, carrying the hyperparameter selected on held-out text. This is
   standard practice: tune on a split, ship on everything.
2. **The model is data + a thin class, not pickled code.** Counts are saved as
   plain JSON (`language_model.json`) and the logic lives in
   `app/language_model.py` — a small `BigramLM` class that loads the JSON and
   answers probability queries. Plain-text artefacts stay inspectable,
   diff-able, and safe to reload anywhere, which a pickle is not.

The export cell writes both, reloads them, and asserts the reloaded model
reproduces this notebook's probabilities exactly.

In [6]:
# ── Build final counts, export model JSON + BigramLM module ──────
BEST_K = 0.005

final_uni = Counter()
final_bi  = Counter()
for sent in sentences:                      # full corpus — split served its purpose
    final_uni.update(sent)
    final_bi.update(zip(sent[:-1], sent[1:]))

LM_PATH = DATA_DIR / "language_model.json"
lm_payload = {
    "k": BEST_K,
    "bos": BOS, "eos": EOS, "unk": UNK,
    "heldout_perplexity": 311.8,
    "unigrams": dict(final_uni),
    "bigrams": {f"{v} {w}": c for (v, w), c in final_bi.items()},
}
LM_PATH.write_text(json.dumps(lm_payload), encoding="utf-8")

MODULE_SOURCE = '''"""
Bigram language model for SciSpell.
Developed and documented in notebooks/03_Language_Model.ipynb — that notebook is
the source of truth; edit there and re-export.
"""
import json
import math
from pathlib import Path

class BigramLM:
    """Add-k smoothed bigram model loaded from language_model.json."""

    def __init__(self, model_path):
        payload = json.loads(Path(model_path).read_text(encoding="utf-8"))
        self.k    = payload["k"]
        self.bos  = payload["bos"]
        self.eos  = payload["eos"]
        self.unk  = payload["unk"]
        self.uni  = payload["unigrams"]
        self.bi   = payload["bigrams"]
        self.V    = len(self.uni) + 1                      # +1 for <UNK>
        self.N    = sum(self.uni.values())

    def _norm(self, w):
        return w if w in self.uni else self.unk

    def p_unigram(self, w):
        """Add-k smoothed unigram probability P(w)."""
        return (self.uni.get(self._norm(w), 0) + self.k) / (self.N + self.k * self.V)

    def p_bigram(self, prev, w):
        """Add-k smoothed bigram probability P(w | prev)."""
        prev, w = self._norm(prev), self._norm(w)
        return ((self.bi.get(prev + " " + w, 0) + self.k)
                / (self.uni.get(prev, 0) + self.k * self.V))

    def logp_sentence(self, tokens):
        """Natural-log probability of a token list (markers added here)."""
        seq = [self.bos] + list(tokens) + [self.eos]
        return sum(math.log(self.p_bigram(v, w)) for v, w in zip(seq[:-1], seq[1:]))

    def perplexity(self, sentences):
        """Corpus perplexity over an iterable of token lists."""
        logp, n = 0.0, 0
        for toks in sentences:
            logp += self.logp_sentence(toks)
            n += len(toks) + 1
        return math.exp(-logp / n)
'''
LM_MODULE_PATH = (PROJECT_ROOT / "app" / "language_model.py")
LM_MODULE_PATH.write_text(MODULE_SOURCE, encoding="utf-8")

# ── Round-trip verification ──
import importlib, language_model
importlib.reload(language_model)
lm = language_model.BigramLM(LM_PATH)

def p_final_addk(prev, w):                  # notebook-side reference
    prev = prev if prev in final_uni else UNK
    w    = w    if w    in final_uni else UNK
    return (final_bi[(prev, w)] + BEST_K) / (final_uni[prev] + BEST_K * (len(final_uni) + 1))

probe = [("of", "the"), ("natural", "selection"), (BOS, "the"),
         ("scientific", "method"), ("the", "keyboard"), ("zzz", "qqq")]
checks = [
    ("Model JSON written", LM_PATH.exists(), f"{LM_PATH.stat().st_size/1_048_576:.1f} MB"),
    ("Module written", LM_MODULE_PATH.exists(), LM_MODULE_PATH.name),
    ("Vocabulary carried over", lm.V == len(final_uni) + 1, f"V = {lm.V:,}"),
    ("Probabilities reproduce notebook",
     all(abs(lm.p_bigram(v, w) - p_final_addk(v, w)) < 1e-12 for v, w in probe),
     f"{len(probe)} probes incl. unseen & OOV"),
    ("Bigram rows sum to 1",
     abs(sum(lm.p_bigram("of", w) for w in list(final_uni) + [UNK]) - 1.0) < 1e-6,
     "context 'of'"),
    ("Sensible sentence beats scrambled",
     lm.logp_sentence(tokenize("the origin of species")) >
     lm.logp_sentence(tokenize("species of origin the")), "log-prob order"),
]

width = max(len(c[0]) for c in checks)
print("VERIFICATION\n" + "─" * (width + 30))
for label, ok, detail in checks:
    print(f"  {'PASS' if ok else 'FAIL'}  {label:<{width}}  {detail}")
print("─" * (width + 30))
print(f"  {sum(ok for _, ok, _ in checks)}/{len(checks)} checks passed")
assert all(ok for _, ok, _ in checks)

print(f"\nSpot checks through the exported model:")
for v, w in [("of", "the"), ("natural", "selection"), ("the", "keyboard")]:
    print(f"  P({w}|{v}) = {lm.p_bigram(v, w):.3e}")

VERIFICATION
───────────────────────────────────────────────────────────────
  PASS  Model JSON written                 1.6 MB
  PASS  Module written                     language_model.py
  PASS  Vocabulary carried over            V = 8,898
  PASS  Probabilities reproduce notebook   6 probes incl. unseen & OOV
  PASS  Bigram rows sum to 1               context 'of'
  PASS  Sensible sentence beats scrambled  log-prob order
───────────────────────────────────────────────────────────────
  6/6 checks passed

Spot checks through the exported model:
  P(the|of) = 2.725e-01
  P(selection|natural) = 6.233e-01
  P(keyboard|the) = 3.059e-07


## Summary

| Component | Result |
|---|---|
| Sentence segmentation | 8,316 sentences, mean 27.6 tokens, `<s>`/`</s>` markers |
| Counts | 8,897 unigram types, 83,664 distinct bigrams |
| Bigram coverage | 0.106% of the 79M possible pairs — the case for smoothing |
| Smoothing | Add-k, k = 0.005 selected on held-out data |
| Held-out perplexity | 311.8 (vs 8,897 uniform baseline — a 28× reduction) |
| Export | `data/language_model.json` + `app/language_model.py`, 6/6 verified |

### Design decisions carried forward

1. **Smoothing is not optional.** With 99.9% of possible bigrams unobserved,
   maximum likelihood assigns zero to almost all valid English, and one zero
   destroys an entire sentence score.
2. **The hyperparameter was measured, not chosen.** Add-one costs up to 96% of
   the observed evidence for domain phrases; k = 0.005 won a held-out perplexity
   tournament against six alternatives.
3. **Domain specificity cuts both ways.** P(*selection* | *natural*) = 0.62 is
   knowledge no general model has, while *scientific method* never occurs at all.
   The corpus makes the corrector expert, not universal.

### Next: Notebook 04 — Correction Engine

The pieces are now in place: a dictionary that decides whether a word is wrong,
a distance that measures how plausible each repair is, and a language model that
scores how probable each candidate is in context. Notebook 04 combines them into
the **noisy channel model** —

    ŵ  =  argmax  P(x | w) · P(w)

— generating candidates, ranking them, and extending the system to **real-word
errors**, where the typed word is in the dictionary but wrong for its context.

In [7]:
# ── Project state after Notebook 03 ──────────────────────────────
print("Project state after Notebook 03\n" + "─" * 46)
for folder in ["data", "figures", "app"]:
    print(f"{folder}/")
    for f in sorted((PROJECT_ROOT / folder).iterdir()):
        if f.is_file():
            print(f"   {f.name:<32} {f.stat().st_size/1024:>9.1f} KB")
        elif f.is_dir():
            print(f"   {f.name}/  ({sum(1 for _ in f.iterdir())} files)")
print("─" * 46)
print(f"Language model: k={lm.k}, V={lm.V:,}, held-out perplexity 311.8")
print("Notebook 03 complete ✓")

Project state after Notebook 03
──────────────────────────────────────────────
data/
   corpus_metadata.json                   1.9 KB
   dictionary.txt                      3773.0 KB
   language_model.json                 1672.0 KB
   misspelling_pairs.json               145.9 KB
   raw/  (5 files)
   science_corpus.txt                  1313.8 KB
   word_freq.json                       128.0 KB
figures/
   edit_distance_tables.png              93.5 KB
   zipf_law.png                         338.8 KB
app/
   __init__.py                            0.0 KB
   __pycache__/  (2 files)
   edit_distance.py                       3.1 KB
   language_model.py                      1.7 KB
──────────────────────────────────────────────
Language model: k=0.005, V=8,898, held-out perplexity 311.8
Notebook 03 complete ✓
